# Fine-Tuning Model - Native Ads Detection (Google Colab)

**Penelitian Postdoc**: Pengembangan Agentic AI untuk Deteksi Native Ads

Notebook ini untuk fine-tuning model di Google Colab (gratis, GPU T4/A100).

**Steps**:
1. Setup environment
2. Upload dataset
3. Load model & apply LoRA
4. Train
5. Save & download model

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes torch

In [ ]:
# Check GPU
!nvidia-smi

## 2. Upload Dataset

Upload file `llm_dataset_instruction.json` dari folder `data/`

In [ ]:
from google.colab import files
import json

# Upload dataset
print("Upload file llm_dataset_instruction.json:")
uploaded = files.upload()

# Load dataset
dataset_file = list(uploaded.keys())[0]
with open(dataset_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"\n✅ Loaded {len(data)} samples")
print(f"Sample: {data[0]}")

## 3. Prepare Dataset

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

# Model to fine-tune
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Format data
def format_sample(sample):
    text = f"{sample['instruction']}\n\n{sample['input']}\n\n{sample['output']}"
    return text

# Tokenize
formatted_texts = [format_sample(s) for s in data]

tokenized = tokenizer(
    formatted_texts,
    truncation=True,
    max_length=512,
    padding='max_length',
    return_tensors='pt'
)

# Create dataset
dataset = Dataset.from_dict({
    'input_ids': tokenized['input_ids'],
    'attention_mask': tokenized['attention_mask']
})

# Split train/eval
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Evaluation samples: {len(eval_dataset)}")

## 4. Load Model & Apply LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Load model with 8-bit quantization
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_8bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)

# Prepare for training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ Model ready for training!")

## 5. Training

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./native-ads-mistral-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to="none",
    warmup_steps=100,
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# Start training
print("🚀 Starting training...")
trainer.train()

print("\n✅ Training complete!")

## 6. Save Model

In [ ]:
# Save model
output_dir = "./native-ads-mistral-lora-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Model saved to: {output_dir}")

## 7. Test Model

In [ ]:
from transformers import pipeline

# Load fine-tuned model
generator = pipeline(
    "text-generation",
    model=output_dir,
    tokenizer=tokenizer,
    device_map="auto"
)

# Test
test_prompt = """Klasifikasikan artikel berikut sebagai native advertising atau konten editorial.

Promo spesial BRI di HUT ke-128, diskon hingga Rp1.28 juta untuk berbagai produk perbankan.
"""

result = generator(test_prompt, max_new_tokens=256, do_sample=False)
print("\n" + "="*80)
print("TEST RESULT:")
print("="*80)
print(result[0]['generated_text'])

## 8. Download Model

In [ ]:
# Zip model
!zip -r native-ads-mistral-lora.zip {output_dir}

# Download
from google.colab import files
files.download('native-ads-mistral-lora.zip')

print("✅ Model downloaded! Extract dan gunakan di local machine.")

## Summary

**Model fine-tuned berhasil!**

Untuk menggunakan di local:
1. Extract zip file
2. Copy folder ke `models/native-ads-mistral-lora/`
3. Jalankan: `python run_local_demo.py --model models/native-ads-mistral-lora --url "URL"`